# 00 - Publish the mock Workday

Terraform (`infra/`) created the function app; this notebook ships its code. It zips `mock_workday/`, sends it to the app's deployment endpoint with your Microsoft Entra token (no publishing keys, no Core Tools), and Azure installs the Python packages. Run it after `terraform apply`, and again whenever `function_app.py` changes.

In [ ]:
import io
import os
import time
import zipfile
from pathlib import Path
from urllib.parse import urlparse

import requests
from dotenv import load_dotenv
from azure.identity import AzureCliCredential

ROOT = Path.cwd() if (Path.cwd() / 'samples').is_dir() else Path.cwd().parent
load_dotenv(ROOT / '.env', override=True)
credential = AzureCliCredential(process_timeout=60)
api_url = os.environ['WORKDAY_MOCK_URL']
# The deployment endpoint sits next to the app: <app>.scm.azurewebsites.net
scm = 'https://' + urlparse(api_url).hostname.replace('.', '.scm.', 1)
print(api_url)
print(scm)

## Publish

The same call `az functionapp deployment source config-zip --build-remote` makes: Azure unpacks the zip, installs `requirements.txt`, and restarts the app. It takes a few minutes.

In [ ]:
package = io.BytesIO()
with zipfile.ZipFile(package, 'w') as archive:
    for name in ['function_app.py', 'host.json', 'requirements.txt']:
        archive.write(ROOT / 'mock_workday' / name, name)

headers = {'Authorization': f"Bearer {credential.get_token('https://management.azure.com/.default').token}"}
response = requests.post(f'{scm}/api/publish?RemoteBuild=true', data=package.getvalue(),
                         headers={**headers, 'Content-Type': 'application/zip'})
print('Publish:', response.status_code, response.text)

status = None
while status not in (3, 4):  # 3 failed, 4 succeeded
    time.sleep(15)
    deployment = requests.get(f'{scm}/api/deployments/latest', headers=headers)
    if deployment.ok:
        status = deployment.json().get('status')
        print(deployment.json().get('progress') or deployment.json().get('status_text') or status)
print('Succeeded' if status == 4 else f'Failed, see {scm}/api/deployments/latest/log')

## Check both sides of the lock

Without a token, App Service authentication answers 401 before the code runs. With yours, the mock answers with your worker record, or 404 until you have one: run `mock_workday\set-role.ps1 manager` (or `employee`) to add yourself.

In [ ]:
token = credential.get_token(f"api://{os.environ['WORKDAY_API_APP_ID']}/.default").token
anonymous = requests.get(api_url)
mine = requests.get(api_url, headers={'Authorization': f'Bearer {token}'})
print('Without a token:', anonymous.status_code)
print('With yours:', mine.status_code, mine.text)